# Auditoria diferencial Z80 — Q / SCF / CCF

Executa na branch `audit/z80-q-scf-ccf-mainline`. A auditoria é **não destrutiva**: não altera o código do emulador. Gera um CSV com os estados observados para comparação com uma referência independente. O resultado do `pytest` não substitui a comparação diferencial.

Escopo: SCF/CCF para todos os valores de A e F; sequências com instruções que escrevem/preservam flags; prefixos DD/FD ignorados; estado Q quando disponível. Ao final, é possível carregar um CSV de referência com as mesmas colunas `case_id,a_before,f_before,q_before,a_after,f_after,q_after,pc,cycles`.

In [ ]:
import os, subprocess, sys, json, csv, time, importlib.util
from pathlib import Path

REPO_URL = 'https://github.com/leonardo201800478/sega-emulator.git'
BRANCH = 'audit/z80-q-scf-ccf-mainline'
REPO_DIR = Path('/content/sega-emulator')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Branch:', subprocess.check_output(['git','branch','--show-current'], text=True).strip())
print('Commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('Python:', sys.version)

# O projeto declara Python >=3.14,<3.15. O runtime padrão do Colab pode ser
# 3.13; não falsificamos essa declaração nem instalamos o pacote em modo editável.
# Como o projeto não possui dependências de runtime, importar diretamente de src
# permite executar a auditoria sem alterar pyproject.toml.
sys.path.insert(0, str(REPO_DIR / 'src'))
for package, import_name in [('pytest', 'pytest'), ('ruff', 'ruff')]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', package], check=True)
print('Importação do código-fonte:', REPO_DIR / 'src')


In [ ]:
# Sanidade local: suíte completa e estilo. Uma falha interrompe a auditoria.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'ruff', 'check', '.'], check=True)
subprocess.run([sys.executable, '-m', 'ruff', 'format', '--check', '.'], check=True)

In [ ]:
from sega_emulator.bus.memory import RAM_SIZE, SG1000Ram
from sega_emulator.bus.memory_bus import SG1000MemoryBus
from sega_emulator.cpu.z80 import Z80Cpu

def make_cpu(program: bytes) -> Z80Cpu:
    return Z80Cpu(SG1000MemoryBus(SG1000Ram(bytes(RAM_SIZE)), program))

def run_case(case_id, program, a, f, q=0):
    cpu = make_cpu(bytes(program))
    cpu.a, cpu.f = a, f
    if hasattr(cpu, 'q'):
        cpu.q = q & 0xFF
    a_before, f_before = cpu.a, cpu.f
    q_before = getattr(cpu, 'q', '')
    cycles = 0
    # Executa exatamente a quantidade de instruções definida pelo caso.
    # SCF/CCF simples e prefixados consomem uma chamada; sequências usam o número de bytes/instruções abaixo.
    for _ in range(1):  # uma instrução por vetor exaustivo
        cycles += cpu.step()
    return {'case_id': case_id, 'a_before': a_before, 'f_before': f_before,
            'q_before': q_before, 'a_after': cpu.a, 'f_after': cpu.f,
            'q_after': getattr(cpu, 'q', ''), 'pc': cpu.pc, 'cycles': cycles}

rows = []
started = time.time()
# Varredura exaustiva dos registradores A/F para SCF (37h) e CCF (3Fh).
for opcode_name, opcode in [('SCF', 0x37), ('CCF', 0x3F)]:
    for a in range(256):
        for f in range(256):
            row = run_case(f'{opcode_name};A={a:02X};F={f:02X}', [opcode], a, f)
            rows.append(row)

# Sequências direcionadas: flags modificadas, flags preservadas e DD/FD ignorados.
sequences = [
    ('INC_B_then_SCF', [0x04, 0x37], 0x00, 0x00, 2),
    ('INC_B_then_CCF', [0x04, 0x3F], 0x00, 0x01, 2),
    ('INC_B_NOP_SCF', [0x04, 0x00, 0x37], 0x00, 0x00, 3),
    ('INC_B_NOP_CCF', [0x04, 0x00, 0x3F], 0x00, 0x01, 3),
    ('DD_ignored_SCF', [0xDD, 0x37], 0x28, 0x01, 1),
    ('FD_ignored_SCF', [0xFD, 0x37], 0x28, 0x01, 1),
    ('DD_ignored_CCF', [0xDD, 0x3F], 0x28, 0x01, 1),
    ('FD_ignored_CCF', [0xFD, 0x3F], 0x28, 0x01, 1),
]
for name, program, a, f, steps in sequences:
    cpu = make_cpu(bytes(program)); cpu.a, cpu.f = a, f
    q_before = getattr(cpu, 'q', '')
    a_before, f_before = cpu.a, cpu.f
    cycles = 0
    for _ in range(steps): cycles += cpu.step()
    rows.append({'case_id': name, 'a_before': a_before, 'f_before': f_before,
                 'q_before': q_before, 'a_after': cpu.a, 'f_after': cpu.f,
                 'q_after': getattr(cpu, 'q', ''), 'pc': cpu.pc, 'cycles': cycles})

out = Path('/content/z80_q_scf_ccf_observed.csv')
with out.open('w', newline='', encoding='utf-8') as fobj:
    writer = csv.DictWriter(fobj, fieldnames=list(rows[0].keys()))
    writer.writeheader(); writer.writerows(rows)
print(f'Geradas {len(rows):,} observações em {time.time()-started:.1f}s: {out}')

## Comparação com referência independente

Faça upload de um CSV produzido por uma implementação de referência ou pelo seu harness diferencial anterior. Ele deve conter `case_id` e as colunas de estado correspondentes (`a_after`, `f_after`, `q_after`; `q_after` pode estar ausente se a referência não expuser Q). O notebook não inventa resultados de referência nem trata a própria CPU como oracle.

In [ ]:
from google.colab import files
from IPython.display import display
uploaded = files.upload()  # selecione CSV de referência; cancele/mande arquivo vazio para apenas exportar observações
reference_name = next((name for name in uploaded if name.lower().endswith('.csv') and name != out.name), None)
if reference_name:
    import pandas as pd
    observed = pd.read_csv(out, dtype={'case_id': str})
    reference = pd.read_csv(reference_name, dtype={'case_id': str})
    required = {'case_id', 'f_after'}
    missing = required - set(reference.columns)
    if missing: raise ValueError(f'CSV de referência sem colunas obrigatórias: {sorted(missing)}')
    keys = ['case_id']
    compare_cols = [c for c in ['a_after','f_after','q_after','pc','cycles'] if c in reference.columns and c in observed.columns]
    merged = observed.merge(reference, on=keys, how='outer', suffixes=('_observed','_reference'), indicator=True)
    mismatch = merged['_merge'].ne('both')
    for col in compare_cols:
        left, right = merged[f'{col}_observed'], merged[f'{col}_reference']
        mismatch |= left.fillna(-1).astype(str).ne(right.fillna(-1).astype(str))
    failures = merged[mismatch]
    failures.to_csv('/content/z80_q_scf_ccf_divergences.csv', index=False)
    print(f'Casos observados: {len(observed):,}; referência: {len(reference):,}; divergências: {len(failures):,}')
    display(failures.head(50))
    print('Arquivo de divergências: /content/z80_q_scf_ccf_divergences.csv')
else:
    print('Nenhuma referência carregada. Foi gerado somente o CSV de observações; não há resultado diferencial ainda.')

files.download(str(out))
if reference_name and Path('/content/z80_q_scf_ccf_divergences.csv').exists():
    files.download('/content/z80_q_scf_ccf_divergences.csv')